# How to create a component

This tutorial demonstrates how to create a component feature.

Component feature includes: lightbox import and 3D texture.

## Prerequisites

### Perform imports

In [1]:
import json
from pathlib import Path

In [2]:
from ansys.speos.core import OptProp, Project, Speos, launcher
from ansys.speos.core.component import LightBox, LightBoxFileInstance, Texture3D
from ansys.speos.core.generic.parameters import (
    OptPropParameters,
    SopTypes,
    Texture3DParameters,
    Texture3DPatternOperations,
    VopOpticParameters,
)
from ansys.speos.core.kernel.client import (
    default_docker_channel,
)
from ansys.speos.core.simulation import SimulationDirect
from ansys.speos.core.source import SourceSurface

### Define constants

The constants help ensure consistency and avoid repetition throughout the example.

In [3]:
HOSTNAME = "localhost"
GRPC_PORT = 50098  # Be sure the Speos GRPC Server has been started on this port.
USE_DOCKER = True  # Set to False if you're running this example locally as a Notebook.
FILES = "CameraInputFiles"

### Define helper functions

These helpers keep LightBox output concise for documentation.
Detailed nested content is truncated because it is not the focus of this example.

In [4]:
def _truncate_json_value(value, *, max_depth=3, max_list_items=8, max_dict_items=12, _depth=0):
    """Truncate nested JSON-like data for readable output.

    Parameters
    ----------
    value : Any
        Input value to truncate.
    max_depth : int, default: 3
        Maximum nesting depth to keep.
    max_list_items : int, default: 8
        Maximum number of items kept per list.
    max_dict_items : int, default: 12
        Maximum number of keys kept per dictionary.
    _depth : int, default: 0
        Current recursion depth. This parameter is for internal use.

    Returns
    -------
    Any
        Truncated representation of the input value.
    """
    if _depth >= max_depth:
        if isinstance(value, dict):
            return {"...": f"{len(value)} keys hidden"}
        if isinstance(value, list):
            return [f"... {len(value)} items hidden"]
        return value

    if isinstance(value, dict):
        items = list(value.items())
        out = {
            key: _truncate_json_value(
                sub_value,
                max_depth=max_depth,
                max_list_items=max_list_items,
                max_dict_items=max_dict_items,
                _depth=_depth + 1,
            )
            for key, sub_value in items[:max_dict_items]
        }
        hidden = len(items) - max_dict_items
        if hidden > 0:
            out["..."] = f"{hidden} more keys"
        return out

    if isinstance(value, list):
        out = [
            _truncate_json_value(
                item,
                max_depth=max_depth,
                max_list_items=max_list_items,
                max_dict_items=max_dict_items,
                _depth=_depth + 1,
            )
            for item in value[:max_list_items]
        ]
        hidden = len(value) - max_list_items
        if hidden > 0:
            out.append(f"... {hidden} more items")
        return out

    return value


def print_lightbox_compact(lightbox, *, max_depth=3, max_list_items=8):
    """Print a compact LightBox representation.

    Parameters
    ----------
    lightbox : ansys.speos.core.component.LightBox
        LightBox feature to print.
    max_depth : int, default: 3
        Maximum nesting depth kept in the printed JSON payload.
    max_list_items : int, default: 8
        Maximum number of items shown per list in the printed payload.

    Returns
    -------
    None
        This function prints formatted output and returns nothing.

    Notes
    -----
    The optional ``"local: "`` prefix from ``str(lightbox)`` is preserved.
    """
    out_str = str(lightbox)
    prefix = ""
    payload = out_str
    if out_str.startswith("local: "):
        prefix = "local: "
        payload = out_str[len(prefix) :]

    try:
        payload_dict = json.loads(payload)
    except json.JSONDecodeError:
        # Fallback to default string output if it is not valid JSON.
        print(out_str)
        return

    compact_payload = _truncate_json_value(
        payload_dict,
        max_depth=max_depth,
        max_list_items=max_list_items,
    )
    print(prefix + json.dumps(compact_payload, indent=4))

## Model Setup

### Load assets
The assets used to run this example are available in the
[PySpeos repository](https://github.com/ansys/pyspeos/) on GitHub.

> **Note:** Make sure you have downloaded the simulation assets and set
> ``assets_data_path`` to point to the assets' folder.

In [5]:
if USE_DOCKER:  # Running on the remote server.
    assets_data_path = Path("/app") / "assets"
else:
    assets_data_path = Path("/path/to/your/download/assets/directory")

### Connect to the RPC Server

This Python client connects to a server where the Speos engine
is running as a service. In this example, the server and
client are the same machine. The launch_local_speos_rpc_method can
be used to start a local instance of the service.

In [6]:
if USE_DOCKER:
    speos = Speos(channel=default_docker_channel())
else:
    speos = launcher.launch_local_speos_rpc_server(port=GRPC_PORT)

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.14/site-packages/ansys/tools/common/cyberchannel.py:201: UserWarning: Starting gRPC client without TLS on localhost:50098. This is INSECURE. Consider using a secure connection.
  warn(f"Starting gRPC client without TLS on {target}. This is INSECURE. Consider using a secure connection.")


## Create a new project

The only way to create a lightbox import, is to create it from a project.

In [7]:
p = Project(speos=speos)
print(p)

{
    "name": "",
    "description": "",
    "metadata": {},
    "part_guid": "",
    "sub_scene_anchor_axis_system": [],
    "sources": [],
    "sensors": [],
    "simulations": [],
    "materials": [],
    "scenes": [],
    "textures_3d": []
}


## Create

Create locally

The mention "local: " is added when printing the lightbox

In [8]:
lightbox = p.create_lightbox(
    name="Light Box Import.1",
    lightbox=LightBoxFileInstance(
        file=assets_data_path / "lightbox" / "Light Box Export.2.SPEOSLightBox", password=""
    ),
)
print_lightbox_compact(lightbox)

local: {
    "name": "Light Box Import.1",
    "axis_system": [
        0.0,
        0.0,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        1.0,
        "... 4 more items"
    ],
    "display_name": "",
    "description": "",
    "metadata": {},
    "scene_guid": "",
    "trajectory_file_uri": ""
}


## Push it to the server.

Now that it is committed to the server, the mention "local: " is no more present when printing
the lightbox.

In [9]:
lightbox.commit()
print_lightbox_compact(lightbox)

{
    "name": "Light Box Import.1",
    "metadata": {
        "UniqueId": "04fa31b0-b7e1-4fb0-97c6-8c5012aad124"
    },
    "scene_guid": "75cc4de7-6218-414d-b4a5-9665250c06a1",
    "axis_system": [
        0.0,
        0.0,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        1.0,
        "... 4 more items"
    ],
    "display_name": "",
    "description": "",
    "trajectory_file_uri": "",
    "scene": {
        "name": "Light Box Export.2",
        "description": "From Speos Light Box: /app/assets/lightbox/Light Box Export.2.SPEOSLightBox",
        "part_guid": "2a60413d-accd-460a-879d-63b017488719",
        "sources": [
            {
                "...": "7 keys hidden"
            }
        ],
        "materials": [
            {
                "...": "8 keys hidden"
            }
        ],
        "metadata": {},
        "sub_scene_anchor_axis_system": [],
        "sensors": [],
        "simulations": [],
        "scenes": [],
        "textures_3d": [],
  

## Read

### Lightbox Instance

Properties methods provided are used to retrieve or modify the information of lightbox.

In [10]:
print(lightbox.name)
print(lightbox.axis_system)
print(lightbox.source_paths)
print(lightbox.trajectory_file_uri)
lightbox.axis_system = [100, 50, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1]
print_lightbox_compact(lightbox)

Light Box Import.1
[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]
['Light Box Import.1/Surface.2:1']

{
    "name": "Light Box Import.1",
    "metadata": {
        "UniqueId": "04fa31b0-b7e1-4fb0-97c6-8c5012aad124"
    },
    "scene_guid": "75cc4de7-6218-414d-b4a5-9665250c06a1",
    "axis_system": [
        0.0,
        0.0,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        1.0,
        "... 4 more items"
    ],
    "display_name": "",
    "description": "",
    "trajectory_file_uri": "",
    "scene": {
        "name": "Light Box Export.2",
        "description": "From Speos Light Box: /app/assets/lightbox/Light Box Export.2.SPEOSLightBox",
        "part_guid": "2a60413d-accd-460a-879d-63b017488719",
        "sources": [
            {
                "...": "7 keys hidden"
            }
        ],
        "materials": [
            {
                "...": "8 keys hidden"
            }
        ],
        "metadata": {},
        "sub_scene_anchor_axi

Commit the modification to the server

In [11]:
lightbox.commit()
print_lightbox_compact(lightbox)

{
    "name": "Light Box Import.1",
    "metadata": {
        "UniqueId": "04fa31b0-b7e1-4fb0-97c6-8c5012aad124"
    },
    "scene_guid": "75cc4de7-6218-414d-b4a5-9665250c06a1",
    "axis_system": [
        100.0,
        50.0,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        1.0,
        "... 4 more items"
    ],
    "display_name": "",
    "description": "",
    "trajectory_file_uri": "",
    "scene": {
        "name": "Light Box Export.2",
        "description": "From Speos Light Box: /app/assets/lightbox/Light Box Export.2.SPEOSLightBox",
        "part_guid": "2a60413d-accd-460a-879d-63b017488719",
        "sources": [
            {
                "...": "7 keys hidden"
            }
        ],
        "materials": [
            {
                "...": "8 keys hidden"
            }
        ],
        "metadata": {},
        "sub_scene_anchor_axis_system": [],
        "sensors": [],
        "simulations": [],
        "scenes": [],
        "textures_3d": [],

### Source and Bodies instance inside Lightbox

Find method is provided to select the features, e.g. sources, bodies inside lightbox.

Example source in lightbox:

In [12]:
lightbox_source = lightbox.find(name=".*", name_regex=True, feature_type=SourceSurface)[0]
print(lightbox_source)

{
    "name": "Surface.2:1",
    "metadata": {
        "UniqueId": "141c5293-ef2a-4f6b-873b-5be93edef8a1"
    },
    "source_guid": "f746c627-5fb7-40c4-ade1-3579f02e02fd",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Surface.2:1",
        "surface": {
            "radiant_flux": {
                "radiant_value": 41.275308514008174
            },
            "intensity_guid": "6ca1eb88-1556-4ca5-925f-2dd42b7a22c6",
            "exitance_constant": {
                "geo_paths": [
                    {
                        "geo_path": "Surface_source2:1796887610/face.1:3037138295",
                        "reverse_normal": false
                    }
                ]
            },
            "spectrum_guid": "a9208c0a-5775-45c8-a3a3-f80096e4b022",
            "intensity": {
                "cos": {
                    "N": 1.0,
                    "total_angle": 180.0
                },
                "name": "",
                "description": "

Example material property in lightbox:

In [13]:
lightbox_material = lightbox.find(name=".*", name_regex=True, feature_type=OptProp)[0]
print(lightbox_material)

{
    "name": "Material.2",
    "metadata": {
        "UniqueId": "97114299-2981-4760-9eb4-58a9532ab4b4"
    },
    "geometries": {
        "geo_paths": [
            "Surface_source2:1796887610/face.1:3037138295"
        ]
    },
    "sop_guid": "dd883a15-7299-4735-b224-2516697c43e5",
    "display_name": "",
    "description": "",
    "sop_guids": [],
    "sop": {
        "mirror": {
            "reflectance": 100.0
        },
        "name": "",
        "description": "",
        "metadata": {}
    }
}


## Modify lightbox features

### feature instance modify method

The Lightbox contains features that can be modified via feature commit.

In [14]:
lightbox_source = lightbox.find(name=".*", name_regex=True, feature_type=SourceSurface)[0]
print(lightbox_source)
lightbox_source.flux.value = 30
lightbox_source.commit()
print(lightbox_source)

{
    "name": "Surface.2:1",
    "metadata": {
        "UniqueId": "141c5293-ef2a-4f6b-873b-5be93edef8a1"
    },
    "source_guid": "f746c627-5fb7-40c4-ade1-3579f02e02fd",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Surface.2:1",
        "surface": {
            "radiant_flux": {
                "radiant_value": 41.275308514008174
            },
            "intensity_guid": "6ca1eb88-1556-4ca5-925f-2dd42b7a22c6",
            "exitance_constant": {
                "geo_paths": [
                    {
                        "geo_path": "Surface_source2:1796887610/face.1:3037138295",
                        "reverse_normal": false
                    }
                ]
            },
            "spectrum_guid": "a9208c0a-5775-45c8-a3a3-f80096e4b022",
            "intensity": {
                "cos": {
                    "N": 1.0,
                    "total_angle": 180.0
                },
                "name": "",
                "description": "

### lightbox instance as global modify method

The Lightbox contains features that can be modified via lightbox commit.

In [15]:
lightbox_material = lightbox.find(name=".*", name_regex=True, feature_type=OptProp)[0]
print(lightbox_material)
lightbox_material.sop_mirror.reflectance = 85
print(lightbox_source)
lightbox_source.intensity.set_cos().total_angle = 20
lightbox.commit()
print(lightbox_material)
print(lightbox_source)

{
    "name": "Material.2",
    "metadata": {
        "UniqueId": "97114299-2981-4760-9eb4-58a9532ab4b4"
    },
    "geometries": {
        "geo_paths": [
            "Surface_source2:1796887610/face.1:3037138295"
        ]
    },
    "sop_guid": "dd883a15-7299-4735-b224-2516697c43e5",
    "display_name": "",
    "description": "",
    "sop_guids": [],
    "sop": {
        "mirror": {
            "reflectance": 100.0
        },
        "name": "",
        "description": "",
        "metadata": {}
    }
}
{
    "name": "Surface.2:1",
    "metadata": {
        "UniqueId": "141c5293-ef2a-4f6b-873b-5be93edef8a1"
    },
    "source_guid": "f746c627-5fb7-40c4-ade1-3579f02e02fd",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Surface.2:1",
        "surface": {
            "radiant_flux": {
                "radiant_value": 30.0
            },
            "intensity_guid": "6ca1eb88-1556-4ca5-925f-2dd42b7a22c6",
            "exitance_constant": {
               

In [16]:
# ## Delete
lightbox_source.delete()

## Lightbox sources in simulation

The project contains two lightbox features.
source_paths are in format of lightbox_name/source_name.

In [17]:
p2 = Project(
    speos=speos,
    path=assets_data_path / "lightbox" / "Direct.1.speos",
)
lightboxes = p2.find(name=".*", name_regex=True, feature_type=LightBox)
lightbox_1 = lightboxes[0]
lightbox_2 = lightboxes[1]
print(lightbox_1.source_paths)
print(lightbox_2.source_paths)

['3/Surface.1:258']
['Light Box Import.2/Surface.2:1', 'Light Box Import.2/Ray-file.1:12']


## Adding lightbox sources in simulation

User can add all the sources from lightbox using source_paths method
Alternatively, user can choose to add selected features

In [18]:
sim = p2.find(name=".*", name_regex=True, feature_type=SimulationDirect)[0]
sim.source_paths = lightbox_2.source_paths
sim.commit()

lightbox_source = lightbox_2.find(name=".*", name_regex=True, feature_type=SourceSurface)[0]
sim.source_paths = [lightbox_source]
sim.commit()

## Black Lightbox

The Black Lightbox is a lightbox which does not show any features' information.
All sources, geometries, materials information cannot be viewed and cannot be edited.

In [19]:
lightbox2 = p.create_lightbox(
    name="Light Box Import.2",
    lightbox=LightBoxFileInstance(
        file=assets_data_path / "lightbox" / "BlackLightBox.SPEOSLightBox", password=""
    ),
)
print_lightbox_compact(lightbox2)
print(lightbox2.name)
print(lightbox2.source_paths)

local: {
    "name": "Light Box Import.2",
    "axis_system": [
        0.0,
        0.0,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        1.0,
        "... 4 more items"
    ],
    "display_name": "",
    "description": "",
    "metadata": {},
    "scene_guid": "",
    "trajectory_file_uri": ""
}
Light Box Import.2
[]


Add a black lightbox to simulation needs to add lightbox to the source paths.

In [20]:
lightbox_3 = p2.create_lightbox(
    name="Black Light Box Import",
    lightbox=LightBoxFileInstance(
        file=assets_data_path / "lightbox" / "BlackLightBox.SPEOSLightBox", password=""
    ),
)
lightbox_3.commit()
sim.source_paths = [lightbox_3.name]
sim.commit()

## 3D Texture

A 3D texture repeats a pattern body on a support body of the project geometry,
following a 3D mapping file.

> **Note:** The 3D texture feature requires a Speos server of version 2027 R1 or higher.

### Define a helper to build a cubic body

Both the support body and the pattern body are simple cubes in this example.

In [21]:
def add_cube_faces(target_body, size):
    """Add the six faces of a cube to a body feature.

    Parameters
    ----------
    target_body : ansys.speos.core.body.Body
        Body feature to fill with faces.
    size : float
        Edge length of the cube.

    Returns
    -------
    ansys.speos.core.body.Body
        Body feature filled with the six cube faces.
    """
    s = size
    faces_definition = {
        "bottom": (
            [0, 0, 0, s, 0, 0, s, s, 0, 0, s, 0],
            [0, 0, -1] * 4,
        ),
        "top": (
            [0, 0, s, s, 0, s, s, s, s, 0, s, s],
            [0, 0, 1] * 4,
        ),
        "left": (
            [0, 0, 0, s, 0, 0, s, 0, s, 0, 0, s],
            [0, -1, 0] * 4,
        ),
        "right": (
            [0, s, 0, s, s, 0, s, s, s, 0, s, s],
            [0, 1, 0] * 4,
        ),
        "front": (
            [s, 0, 0, s, s, 0, s, s, s, s, 0, s],
            [1, 0, 0] * 4,
        ),
        "back": (
            [0, 0, 0, 0, s, 0, 0, s, s, 0, 0, s],
            [-1, 0, 0] * 4,
        ),
    }
    for face_name, (vertices, normals) in faces_definition.items():
        new_face = target_body.create_face(name=face_name)
        new_face.vertices = vertices
        new_face.facets = [0, 1, 2, 0, 2, 3]
        new_face.normals = normals
    return target_body

### Create the support geometry

The support body must be a volume body: a volume optical property is assigned to it.

In [22]:
p3 = Project(speos=speos)

root_part = p3.create_root_part()
support_body = add_cube_faces(root_part.create_body(name="SupportBody"), size=10)
support_body.commit()
root_part.commit()

support_material = p3.create_optical_property(
    name="SupportMaterial",
    parameters=OptPropParameters(
        vop_parameters=VopOpticParameters(index=1.5, absorption=0.0),
        sop_parameters=SopTypes.optical_polished,
    ),
)
support_material.geometries = [support_body]
support_material.commit()

### Create the 3D texture locally

Like any other feature, the 3D texture is first created locally.
The mention "local: " is added when printing it.

Default values can be overwritten at creation time with a
``Texture3DParameters`` dataclass.

In [23]:
texture_3d = p3.create_3d_texture(
    name="3D Texture.1",
    parameters=Texture3DParameters(global_scale=1.0),
)
print(texture_3d.axis_system)
print(texture_3d.pattern_operation)
print(texture_3d.global_scale)
print(texture_3d)

[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]
Texture3DPatternOperations.add_on_same_material
1.0
local: {
    "axis_system": [
        0.0,
        0.0,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        1.0
    ],
    "pattern_operation": "PATTERN_OPERATION_ADD_ON_SAME_MATERIAL",
    "global_scale": 1.0,
    "support_body_geo_path": "",
    "mapping_3d_file_uri": ""
}


### Define the pattern body

The pattern body is created from the 3D texture itself. It is automatically attached
to the 3D texture when the 3D texture is committed.

> **Note:** The pattern body must not be a body of the scene root part taken
> in the simulation.

In [24]:
pattern_body = texture_3d.create_pattern_body()
pattern_body = add_cube_faces(pattern_body, size=1)
pattern_body.commit()

### Add a material on the pattern

This material belongs to the 3D texture: it is not registered as a project feature.

In [25]:
pattern_material = texture_3d.create_pattern_material(
    name="PatternMaterial",
    parameters=OptPropParameters(
        vop_parameters=VopOpticParameters(index=1.6, absorption=0.0),
        sop_parameters=SopTypes.optical_polished,
    ),
)
pattern_material.geometries = [pattern_body]
pattern_material.commit()

### Configure the 3D texture

The support body, the pattern operation, the global scale and the 3D mapping file
are set through properties. The pattern operations ``add_on_different_material``,
``add_in`` and ``insert`` etc the material of the pattern body.

In [26]:
texture_3d.axis_system = [10, 20, 30, 1, 0, 0, 0, 1, 0, 0, 0, 1]
texture_3d.support_body = support_body
texture_3d.pattern_operation = Texture3DPatternOperations.add_in
texture_3d.global_scale = 2.5
texture_3d.mapping_3d_file_uri = (
    assets_data_path / "texture3d" / "Component1.3D Texture.1.OPT3DMapping"
)

### Push it to the server

Once committed, the mention "local: " is no more present when printing the 3D texture.

In [27]:
texture_3d.commit()
print(texture_3d)

{
    "axis_system": [
        10.0,
        20.0,
        30.0,
        1.0,
        0.0,
        0.0,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        1.0
    ],
    "support_body_geo_path": "SupportBody",
    "pattern_operation": "PATTERN_OPERATION_ADD_IN",
    "pattern": {
        "body_guid": "a132d211-9a18-42f0-893e-21b161afa8a8",
        "materials": [
            {
                "name": "PatternMaterial",
                "metadata": {
                    "UniqueId": "20343f52-6f9f-48e1-88c2-5df9c5211a29"
                },
                "vop_guid": "225726f8-b1e7-417a-9530-ae8e227ba3a0",
                "geometries": {
                    "geo_paths": [
                        "3D Texture.1"
                    ]
                },
                "sop_guid": "c0adbe16-e3fb-44e6-b7e6-828a16bc1be1",
                "display_name": "",
                "description": "",
                "sop_guids": [],
                "vop": {
                    "name

### Read

Properties are used to retrieve the 3D texture information, and the ``get`` method
gives a read-only access to the server data.

In [28]:
print(texture_3d.name)
print(texture_3d.support_body)
print(texture_3d.pattern_operation)
print(texture_3d.global_scale)
print(texture_3d.mapping_3d_file_uri)
print([material._name for material in texture_3d.pattern_materials])
print(texture_3d.get(key="global_scale"))

3D Texture.1
SupportBody
Texture3DPatternOperations.add_in
2.5
/app/assets/texture3d/Component1.3D Texture.1.OPT3DMapping
['PatternMaterial']
2.5


### Update

Modify the local data, then commit to push the modification to the server.

In [29]:
texture_3d.global_scale = 1.5
texture_3d.commit()
print(texture_3d.global_scale)

1.5


The ``reset`` method discards the local modifications that were not committed.

In [30]:
texture_3d.global_scale = 9.0
texture_3d.reset()
print(texture_3d.global_scale)

1.5


### Find a 3D texture in a project

A project loaded from a ``.speos`` file exposes its 3D textures like any other feature.

In [31]:
textures_3d = p3.find(name=".*", name_regex=True, feature_type=Texture3D)
print(len(textures_3d))
print(textures_3d[0].name)

1
3D Texture.1


### Delete

Deleting the 3D texture also deletes its pattern body and the templates of its
pattern materials.

In [32]:
texture_3d.delete()